# Part 1: Data extraction

## AI PR tables

Downloads the AIDev v3 tables for AI PRs (repositories with more than 100 stars) from Hugging Face into `data/`.

In [ ]:
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download

# Download v3 Parquet files (AI PRs)
DATASET_REPO = "hao-li/AIDev"
VERSION = "v3"
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# human_pull_request.parquet is no longer downloaded: the instructor's Human PR
# tables (next cell) contain the same 6,618 PRs with the AIDev-pop layout.
parquet_files = {
    "pull_request_df": "pull_request.parquet",
    "pr_task_type_df": "pr_task_type.parquet",
    "repository_df": "repository.parquet",
    #"user_df": "user.parquet",
    "pr_timeline_df": "pr_timeline.parquet",
    "pr_reviews_df": "pr_reviews.parquet",
    "pr_comments_df": "pr_comments.parquet",
    "pr_review_comments_df": "pr_review_comments_v2.parquet",
    "pr_commits_df": "pr_commits.parquet",
    "pr_commit_details_df": "pr_commit_details.parquet",
    "issue_df": "issue.parquet",
    "related_issue_df": "related_issue.parquet"
}

ai_dataframes = {}

for dataframe_name, filename in parquet_files.items():
    local_path = hf_hub_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        filename=filename,
        revision=VERSION,
        local_dir=DATA_DIR,
    )
    ai_dataframes[dataframe_name] = pd.read_parquet(local_path)
    print(f"Loaded {filename} from {local_path}")

print("AIDev v3 files loaded")

## Human PR tables

Loads the Human PR tables given by the instructor. Copy the `AIDev_Human_PRs` folder into `data/` before running this cell (it is not on Hugging Face and is too large for GitHub).

In [ ]:
HUMAN_DATA_DIR = DATA_DIR / "AIDev_Human_PRs"

human_parquet_files = {
    "pull_request_df": "pull_request.parquet",
    "pr_task_type_df": "pr_task_type.parquet",
    "repository_df": "repository.parquet",
    "pr_timeline_df": "pr_timeline.parquet",
    "pr_reviews_df": "pr_reviews.parquet",
    "pr_comments_df": "pr_comments.parquet",
    "pr_review_comments_df": "pr_review_comments.parquet",
    "pr_commits_df": "pr_commits.parquet",
    "pr_commit_details_df": "pr_commit_details.parquet",
    "issue_df": "issue.parquet",
    "related_issue_df": "related_issue.parquet"
}

human_dataframes = {}

for dataframe_name, filename in human_parquet_files.items():
    local_path = HUMAN_DATA_DIR / filename
    human_dataframes[dataframe_name] = pd.read_parquet(local_path)
    print(f"Loaded {filename} from {local_path}")

print("Human PR files loaded")

## Combine AI and Human tables

Each Human table is aligned to its AI version (same columns, order and types), then added to it. After this cell, `dataframes` has one table per entity with both groups, and `pull_request_df.agent == "Human"` marks Human PRs.

Differences fixed on the way:

| Table | Difference | Fix |
| --- | --- | --- |
| `related_issue_df` | AI `issue_id` stored as a decimal number | Converted to an integer, like `issue_df.id` |
| `pr_task_type_df` | Human PRs have no `confidence` | Left empty for Human PRs |
| `pr_review_comments_df` | `position` has a different type | Human converted to the AI type |
| `repository_df` | Human has an extra `is_forked` column | Dropped |
| `pr_reviews_df` | Human table has one row per review **comment**, with no `state` or `submitted_at` | Grouped to one row per review, like the AI table. `state`, `submitted_at` and `body` stay empty |

812 repositories and 30 issues appear in both groups. We keep the AI row for those (star counts differ slightly because the data was collected at different times).

In [ ]:
def align_to(reference_df, df):
    """Return df with the columns, column order and dtypes of reference_df."""
    extra_columns = set(df.columns) - set(reference_df.columns)
    assert not extra_columns, f"Unexpected columns: {extra_columns}"
    return df.reindex(columns=reference_df.columns).astype(reference_df.dtypes.to_dict())


ai_dataframes["related_issue_df"]["issue_id"] = ai_dataframes["related_issue_df"]["issue_id"].astype("int64")
ai_dataframes["pr_task_type_df"]["confidence"] = ai_dataframes["pr_task_type_df"]["confidence"].astype("Int64")

human_dataframes["pr_review_comments_df"]["position"] = human_dataframes["pr_review_comments_df"]["position"].astype("float64")
human_dataframes["repository_df"] = human_dataframes["repository_df"].drop(columns="is_forked")
human_dataframes["pr_reviews_df"] = (
    human_dataframes["pr_reviews_df"]
    .groupby("pull_request_review_id", as_index=False)
    .agg(pr_id=("pr_id", "first"), user=("user", "first"), user_type=("user_type", "first"))
    .rename(columns={"pull_request_review_id": "id"})
)

# Tables keyed by id that describe entities shared by both groups: keep the AI row.
shared_entity_tables = ["repository_df", "issue_df"]

dataframes = {}

for dataframe_name, ai_df in ai_dataframes.items():
    human_df = align_to(ai_df, human_dataframes[dataframe_name])
    combined_df = pd.concat([ai_df, human_df], ignore_index=True)
    if dataframe_name in shared_entity_tables:
        combined_df = combined_df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
    dataframes[dataframe_name] = combined_df
    print(f"{dataframe_name:25} AI: {len(ai_df):7}  Human: {len(human_df):7}  Combined: {len(combined_df):7}")

for dataframe_name, dataframe in dataframes.items():
    assert dataframe.dtypes.equals(ai_dataframes[dataframe_name].dtypes), dataframe_name

print("AI and Human tables combined")

Quick look at the combined tables: number of rows and columns, and the first rows of each table.

In [ ]:

def table_inventory(dataframes):
    rows = []
    for table_name, dataframe in dataframes.items():
        rows.append(
            {
                "table": table_name,
                "rows": len(dataframe),
                "columns": len(dataframe.columns),
                "column_names": ", ".join(dataframe.columns),
            }
        )
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)

inventory = table_inventory(dataframes)
# pd.set_option("display.max_colwidth", 160)
display(inventory)

for table_name, dataframe in dataframes.items():
    print(f"\n=== {table_name} ===")
    display(dataframe.head())

# Part 2: Data cleaning and classification

### Q1: How you removed duplicates

1. **Same PR in both groups.** 56 Human PRs are also in the AI data (same `html_url`, different `id`). They are agent PRs, so we kept the AI version and removed the Human copy, with all its linked rows (commits, files, reviews, comments, timeline, linked issues).
2. **Rows that appear twice.** We removed exact copies:

| Table | Rows removed | Note |
| --- | --- | --- |
| `pr_commits_df` | 12 | |
| `pr_commit_details_df` | 23 | Rows where a file was removed and added back in the same commit are kept |
| `related_issue_df` | 50 | Some Human PR–issue links were found twice (in the PR description and in the timeline). We keep one, from the description, the only source used for AI PRs |
| `pr_timeline_df` | 12 | Only repeated `committed` events. Other identical rows can be real separate events (the timeline has no event id), so they are kept |

The code then checks that no id appears twice in the tables that have one.

### Q2: How you handled missing values

Most missing values have a meaning, so we kept them instead of guessing a value:

- **Dates** are converted to UTC. An empty `closed_at` means the PR or issue is still open. An empty `merged_at` means the PR was not merged. The code checks that this holds for every PR and issue.
- **Text** (PR or issue description, review text, commit message): empty values become `""`, since nothing was written.
- **Left empty on purpose:** fields missing from the Human data (`confidence`, review `state` and `submitted_at`), commits that changed no files, very large diffs that GitHub does not return (`patch`), timeline columns that only apply to some events, and repository `license` / `language` (not used).
- **29 PRs have no commit data.** They are kept for merge rate and time to merge, but will be left out of the size metrics.

After cleaning: 33,596 AI PRs and 6,562 Human PRs.

In [ ]:
cleaning_log = []


def log_step(step, table_name, rows_before, rows_after):
    cleaning_log.append({"step": step, "table": table_name, "rows_removed": rows_before - rows_after, "rows_after": rows_after})


# Treat Human PRs that are AIDev agent PRs (same html_url, different id): keep the AI version and drop
# the Human copy with all its child rows.
pull_request_df = dataframes["pull_request_df"]
is_human = pull_request_df["agent"] == "Human"
ai_urls = set(pull_request_df.loc[~is_human, "html_url"])
duplicate_pr_ids = set(pull_request_df.loc[is_human & pull_request_df["html_url"].isin(ai_urls), "id"])
duplicate_review_ids = set(dataframes["pr_reviews_df"].loc[dataframes["pr_reviews_df"]["pr_id"].isin(duplicate_pr_ids), "id"])

for dataframe_name, dataframe in dataframes.items():
    if dataframe_name in shared_entity_tables:
        continue
    if dataframe_name in ["pull_request_df", "pr_task_type_df"]:
        is_duplicate_pr = dataframe["id"].isin(duplicate_pr_ids)
    elif dataframe_name == "pr_review_comments_df":
        is_duplicate_pr = dataframe["pull_request_review_id"].isin(duplicate_review_ids)
    else:
        is_duplicate_pr = dataframe["pr_id"].isin(duplicate_pr_ids)
    dataframes[dataframe_name] = dataframe[~is_duplicate_pr].reset_index(drop=True)
    log_step("Human copy of an agent PR", dataframe_name, len(dataframe), len(dataframes[dataframe_name]))

# Exact duplicate rows treatment
for dataframe_name, dataframe in dataframes.items():
    if dataframe_name == "pr_timeline_df":
        is_duplicate = (dataframe["event"] == "committed") & dataframe.duplicated(subset=["pr_id", "event", "commit_id"])
    elif dataframe_name == "related_issue_df":
        dataframe = dataframe.sort_values("source", kind="stable")
        is_duplicate = dataframe.duplicated(subset=["pr_id", "issue_id"])
    else:
        is_duplicate = dataframe.duplicated()
    dataframes[dataframe_name] = dataframe[~is_duplicate].reset_index(drop=True)
    log_step("Exact duplicate row", dataframe_name, len(dataframe), len(dataframes[dataframe_name]))

primary_keys = {
    "pull_request_df": ["id"],
    "pr_task_type_df": ["id"],
    "repository_df": ["id"],
    "issue_df": ["id"],
    "pr_reviews_df": ["id"],
    "pr_comments_df": ["id"],
    "pr_review_comments_df": ["id"],
    "pr_commits_df": ["sha", "pr_id"],
    "related_issue_df": ["pr_id", "issue_id"],
}
for dataframe_name, key_columns in primary_keys.items():
    assert not dataframes[dataframe_name].duplicated(subset=key_columns).any(), f"Duplicate key in {dataframe_name}"

# Treat missing timestamps
timestamp_columns = {
    "pull_request_df": ["created_at", "closed_at", "merged_at"],
    "issue_df": ["created_at", "closed_at"],
    "pr_timeline_df": ["created_at"],
    "pr_reviews_df": ["submitted_at"],
    "pr_comments_df": ["created_at"],
    "pr_review_comments_df": ["created_at", "updated_at"],
}
for dataframe_name, columns in timestamp_columns.items():
    for column in columns:
        parsed = pd.to_datetime(dataframes[dataframe_name][column], utc=True, errors="coerce")
        assert parsed.notna().equals(dataframes[dataframe_name][column].notna()), f"Unparsable dates in {dataframe_name}.{column}"
        dataframes[dataframe_name][column] = parsed

# Treat missing text (replace with empty strings)
text_columns = {
    "pull_request_df": ["body"],
    "issue_df": ["body"],
    "pr_reviews_df": ["body"],
    "pr_review_comments_df": ["body"],
    "pr_commits_df": ["message"],
    "pr_commit_details_df": ["message"],
}
for dataframe_name, columns in text_columns.items():
    dataframes[dataframe_name][columns] = dataframes[dataframe_name][columns].fillna("")

# Missing timestamps must match the PR and issue states
pull_request_df = dataframes["pull_request_df"]
assert pull_request_df["closed_at"].isna().equals(pull_request_df["state"] == "open")
assert pull_request_df.loc[pull_request_df["merged_at"].notna(), "closed_at"].notna().all()
assert (pull_request_df["merged_at"].dropna() >= pull_request_df.loc[pull_request_df["merged_at"].notna(), "created_at"]).all()
assert (pull_request_df["closed_at"].dropna() >= pull_request_df.loc[pull_request_df["closed_at"].notna(), "created_at"]).all()
assert dataframes["issue_df"]["closed_at"].isna().equals(dataframes["issue_df"]["state"] == "open")

cleaning_summary = pd.DataFrame(cleaning_log)
display(cleaning_summary[cleaning_summary["rows_removed"] > 0].reset_index(drop=True))

remaining_missing = pd.DataFrame(
    [
        {"table": dataframe_name, "column": column, "missing": count, "pct": round(100 * count / len(dataframe), 2)}
        for dataframe_name, dataframe in dataframes.items()
        for column, count in dataframe.isna().sum().items()
        if count > 0
    ]
)
display(remaining_missing)

prs_without_commits = ~pull_request_df["id"].isin(dataframes["pr_commits_df"]["pr_id"])
print(f"PRs without commit data (kept, excluded later from size metrics): {prs_without_commits.sum()}")
print(pull_request_df["agent"].eq("Human").map({True: "Human", False: "AI"}).value_counts().to_string())

### Q3: How you identified AI-related vs. human-related contributions

**AI PRs:** we used the AIDev labels (`agent` column). We checked that each agent's trace is really in the data: Claude Code footer in the description (100% of its PRs), PR opened by `devin-ai-integration[bot]` (100%) or `Copilot` (100%), Codex link in the description (80%), commits by `cursoragent` (97%).

**Human PRs:** the Human data is not all human, so the code looks at who opened each PR and for AI traces in the description and commits. 

How the accounts are found:
- **Bots:** a login ending with `[bot]` or `bot` (the same rule is used for reviews and comments). One person whose login ends with `bot` is kept as human.
- **Accounts of the five AIDev agents:** found in the data, as the accounts that open or commit in more than half of an agent's PRs (`Copilot`, `cursoragent`, `devin-ai-integration[bot]`). Claude Code and Codex use the developer's account.
- **Other AI agents and AI review bots:** no column says whether a bot is an AI, so this part is manual. The code prints the distinct bot accounts found in the data (PR authors, committers, reviewers and commenters), the decisions are made from that list, and the code checks that every account in the decisions is in it.

The result is the `ai_involvement` column:

| `ai_involvement` | Meaning | PRs |
| --- | --- | --- |
| `human_only` | Human PR, no AI trace | 5,436 |
| `human_with_ai` | Human PR with some commits written by an agent | 129 |
| `agent_unlabeled` | Agent PR that AIDev missed (agent trace in the description) | 81 |
| `bot` | Opened by a bot (dependabot, renovate, ...) | 885 |
| `other_ai` | Opened by an AI agent that is not one of the 5 AIDev agents | 31 |
| `agent` | AIDev agent PR | 33,596 |

The code also adds:
- `has_human_commits`: AI PR where a person also pushed commits (29% of Copilot, 31% of Devin and 41% of Cursor PRs). Not measurable for Codex and Claude Code, which commit under the developer's account.
- `actor_type` on reviews and comments: `human`, `ai_bot` or `other_bot`. 40% of reviews on AI PRs come from AI bots.
- `in_matched_subset`: AI PRs from the same repositories and period as the Human PRs (7,541), for a fair comparison.

See *Notes for Parts 3 and 4* at the end of the notebook for how to use these columns.

In [ ]:
pull_request_df = dataframes["pull_request_df"]
pr_commits_df = dataframes["pr_commits_df"]

pull_request_df["group"] = pull_request_df["agent"].where(pull_request_df["agent"] == "Human", "AI")
is_human_pr = pull_request_df["group"] == "Human"
human_pr_ids = pull_request_df.loc[is_human_pr, "id"]
AIDEV_AGENTS = sorted(pull_request_df.loc[~is_human_pr, "agent"].unique())

# A login ending with "[bot]" or "bot" is a bot account (same rule as for reviews and comments below)
BOT_LOGIN_PATTERN = r"(?:\[bot\]|bot)$"


def is_bot_login(logins):
    return logins.str.contains(BOT_LOGIN_PATTERN, case=False, regex=True, na=False)


# --- Accounts of the five AIDev agents, found in the data ---
# An account belongs to an agent if it opens, or authors commits in, more than half of the agent's PRs.
# Claude Code and Codex work under the developer's account, so they get no account.
commit_agent = pr_commits_df["pr_id"].map(pull_request_df.set_index("id")["agent"])
AGENT_OWN_ACCOUNTS = {}
for agent in AIDEV_AGENTS:
    agent_prs = pull_request_df[pull_request_df["agent"] == agent]
    agent_commits = pr_commits_df[commit_agent == agent]
    opener_share = agent_prs["user"].value_counts(normalize=True)
    author_share = agent_commits.groupby("author")["pr_id"].nunique() / agent_commits["pr_id"].nunique()
    AGENT_OWN_ACCOUNTS[agent] = set(opener_share[opener_share > 0.5].index) | set(author_share[author_share > 0.5].index)
print("AIDev agent accounts found in the data:", AGENT_OWN_ACCOUNTS)
AIDEV_AGENT_ACCOUNTS = set().union(*AGENT_OWN_ACCOUNTS.values())

# --- Candidates for the manual decisions below: distinct bot accounts in the data ---
human_pr_authors = pull_request_df.loc[is_human_pr, "user"]
bot_authors = human_pr_authors[is_bot_login(human_pr_authors)].value_counts()
print(f"\nBot-like authors of Human PRs ({len(bot_authors)}), with their number of PRs:")
print(bot_authors.to_string())

human_pr_commits = pr_commits_df[pr_commits_df["pr_id"].isin(human_pr_ids)]
committers = pd.concat([human_pr_commits["author"], human_pr_commits["committer"]])
bot_committers = committers[is_bot_login(committers)].value_counts()
print(f"\nBot-like commit authors and committers in Human PRs ({len(bot_committers)}), with their number of commits:")
print(bot_committers.to_string())

reviewers = pd.concat([dataframes[name][["user", "user_type"]] for name in ["pr_reviews_df", "pr_comments_df", "pr_review_comments_df"]])
bot_reviewers = reviewers.loc[(reviewers["user_type"] == "Bot") | is_bot_login(reviewers["user"]), "user"].value_counts()
print(f"\nBot accounts that review or comment ({len(bot_reviewers)}); the 60 most active, with their number of reviews and comments:")
print(bot_reviewers.head(60).to_string())

# --- Manual decisions, made by reading the candidates printed above ---
# Logins that end with "bot" but belong to a person (checked on their PRs)
NOT_BOT_LOGINS = {"connortbot"}
# AI coding agents that are not one of the five AIDev agents (bot authors and committers above)
OTHER_AI_ACCOUNTS = {
    "gru-agent[bot]", "codeflash-ai[bot]", "tembo-io[bot]", "dane-ai-mastra[bot]",
    "entelligence-ai-pr-reviews[bot]", "RepoBirdBot", "claude[bot]", "google-labs-jules[bot]",
}
# AI code review bots (bot reviewers and commenters above)
AI_REVIEW_BOTS = {
    "copilot-swe-agent[bot]", "copilot-pull-request-reviewer[bot]", "cursor[bot]", "coderabbitai[bot]",
    "greptile-apps[bot]", "gemini-code-assist[bot]", "qodo-merge-for-open-source[bot]", "qodo-merge-pro[bot]",
    "ellipsis-dev[bot]", "cubic-dev-ai[bot]", "sourcery-ai[bot]", "recurseml[bot]", "seer-by-sentry[bot]",
    "jazzberry-ai[bot]", "kodus-ai[bot]", "use-tusk[bot]",
}
# Every decision must refer to an account that is really in the candidates
assert NOT_BOT_LOGINS <= set(bot_authors.index)
assert OTHER_AI_ACCOUNTS <= set(bot_authors.index) | set(bot_committers.index)
assert AI_REVIEW_BOTS <= set(bot_reviewers.index)

# --- Who opened each PR ---
author_type = pd.Series("agent", index=pull_request_df.index)
author_type[is_human_pr] = "human"
author_type[is_human_pr & is_bot_login(pull_request_df["user"]) & ~pull_request_df["user"].isin(NOT_BOT_LOGINS)] = "bot"
author_type[is_human_pr & pull_request_df["user"].isin(OTHER_AI_ACCOUNTS)] = "other_ai"
pull_request_df["author_type"] = author_type

# Signatures the agents leave in PR descriptions and commit messages (PRs co-authored Human-AI)
AI_SIGNATURE_PATTERN = (
    r"Generated with \[?Claude Code|claude\.ai/code|Co-Authored-By: Claude|copilot-swe-agent|"
    r"Co-authored-by: Copilot|chatgpt\.com/codex|cursoragent|devin\.ai|app\.devin"
)
# Accounts that agents use to author or commit code
AGENT_COMMIT_ACCOUNTS = AIDEV_AGENT_ACCOUNTS | OTHER_AI_ACCOUNTS

has_description_trace = (
    pull_request_df["title"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
    | pull_request_df["body"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
)
is_ai_commit = (
    pr_commits_df["message"].str.contains(AI_SIGNATURE_PATTERN, case=False, regex=True)
    | pr_commits_df["author"].isin(AGENT_COMMIT_ACCOUNTS)
    | pr_commits_df["committer"].isin(AGENT_COMMIT_ACCOUNTS)
)
has_commit_trace = pull_request_df["id"].isin(pr_commits_df.loc[is_ai_commit, "pr_id"])

# AI involvement level of every PR

ai_involvement = author_type.copy()
ai_involvement[author_type == "human"] = "human_only"
ai_involvement[(author_type == "human") & has_commit_trace] = "human_with_ai"
ai_involvement[(author_type == "human") & has_description_trace] = "agent_unlabeled"
pull_request_df["ai_involvement"] = ai_involvement

# Human commits in AI PRs (only measurable for agents that have their own account)
measurable_agents = [agent for agent, accounts in AGENT_OWN_ACCOUNTS.items() if accounts]
is_human_commit = pd.Series(False, index=pr_commits_df.index)
for agent in measurable_agents:
    is_human_commit |= (
        (commit_agent == agent)
        & pr_commits_df["author"].notna()
        & ~pr_commits_df["author"].isin(AGENT_OWN_ACCOUNTS[agent])
        & ~pr_commits_df["author"].str.endswith("[bot]", na=False)
    )

has_commits = pull_request_df["id"].isin(pr_commits_df["pr_id"])
has_human_commits = pd.Series(pd.NA, index=pull_request_df.index, dtype="boolean")
measurable = pull_request_df["agent"].isin(measurable_agents) & has_commits
has_human_commits[measurable] = pull_request_df.loc[measurable, "id"].isin(pr_commits_df.loc[is_human_commit, "pr_id"])
pull_request_df["has_human_commits"] = has_human_commits

# Matched subset: AI PRs from the same repositories and period as the Human PRs

human_repo_ids = set(pull_request_df.loc[is_human_pr, "repo_id"])
human_period_start = pull_request_df.loc[is_human_pr, "created_at"].min()
human_period_end = pull_request_df.loc[is_human_pr, "created_at"].max()
pull_request_df["in_matched_subset"] = (
    pull_request_df["repo_id"].isin(human_repo_ids)
    & pull_request_df["created_at"].between(human_period_start, human_period_end)
)


# AI agents and AI code review bots
AI_BOT_LOGINS = AIDEV_AGENT_ACCOUNTS | OTHER_AI_ACCOUNTS | AI_REVIEW_BOTS


def classify_actor(dataframe):
    is_bot = (dataframe["user_type"] == "Bot") | is_bot_login(dataframe["user"])
    actor_type = pd.Series("human", index=dataframe.index)
    actor_type[is_bot] = "other_bot"
    actor_type[dataframe["user"].isin(AI_BOT_LOGINS)] = "ai_bot"
    actor_type[dataframe["user"].isna()] = pd.NA
    return actor_type


for dataframe_name in ["pr_reviews_df", "pr_comments_df", "pr_review_comments_df"]:
    dataframes[dataframe_name]["actor_type"] = classify_actor(dataframes[dataframe_name])

display(pd.crosstab(pull_request_df["ai_involvement"], pull_request_df["group"], margins=True))
display(
    pull_request_df[pull_request_df["group"] == "AI"]
    .groupby("agent")
    .agg(
        prs=("id", "size"),
        in_matched_subset=("in_matched_subset", "sum"),
        with_human_commits=("has_human_commits", "sum"),
        human_commits_measured=("has_human_commits", "count"),
    )
)
pr_group = pull_request_df.set_index("id")["group"]
display(
    pd.concat(
        {
            dataframe_name: pd.crosstab(
                dataframes[dataframe_name]["pr_id"].map(pr_group), dataframes[dataframe_name]["actor_type"], normalize="index"
            ).round(3)
            for dataframe_name in ["pr_reviews_df", "pr_comments_df"]
        }
    )
)

### Q4: How you classified task types

1. **Task type of each PR.** The AIDev dataset already gives every PR, AI and Human, one type (`feat`, `fix`, `docs`, `test`, ...), chosen by an LLM from the PR title and description. We copy it into `pull_request_df` as `task_type`.
2. **Categories from the lab instructions.** We group the types into the `task_category` column:

| AIDev type | Category |
| --- | --- |
| `fix` | Bug fixes |
| `feat` | Feature implementation |
| `docs` | Documentation |
| `refactor`, `style`, `perf` | Refactoring |
| `test` | Testing |
| `ci`, `build` | CI/CD or configuration |
| `chore`, `revert`, `other` | Maintenance |

No AIDev type matches "Experimental or research-related changes", so that category is not used.

In [ ]:
pull_request_df = dataframes["pull_request_df"]
pr_task_type_df = dataframes["pr_task_type_df"]

# Task type from AIDev (LLM classification into Conventional Commit types)
pull_request_df["task_type"] = pull_request_df["id"].map(pr_task_type_df.set_index("id")["type"])
assert pull_request_df["task_type"].notna().all()

# Assignment categories (wording from the lab PDF). No AIDev type matches
# "Experimental or research-related changes", so that category stays empty.
TASK_CATEGORIES = {
    "fix": "Bug fixes",
    "feat": "Feature implementation",
    "docs": "Documentation",
    "refactor": "Refactoring",
    "style": "Refactoring",
    "perf": "Refactoring",
    "test": "Testing",
    "ci": "CI/CD or configuration",
    "build": "CI/CD or configuration",
    "chore": "Maintenance",
    "revert": "Maintenance",
    "other": "Maintenance",
}
pull_request_df["task_category"] = pull_request_df["task_type"].map(TASK_CATEGORIES)
assert pull_request_df["task_category"].notna().all()

display(pd.crosstab(pull_request_df["task_category"], pull_request_df["group"], margins=True))

### Q5: Any assumptions you made

**Data**
- The instructor's Human tables describe the same 6,618 PRs as AIDev's `human_pull_request` table (we checked: same ids and values, except that some descriptions use different line-break characters).
- Human reviews only include reviews that left inline comments, and have no review state.

**Duplicates and missing values**
- Two PRs with the same `html_url` are the same PR. When it is in both groups, the AI label is kept.
- Identical timeline rows can be separate real events, so only repeated `committed` events are removed.
- An empty `closed_at` means "still open", an empty `merged_at` means "not merged", and empty text means "nothing written".

**AI vs. human**
- AIDev's agent labels are correct.
- A Human PR with no visible AI trace was written without AI. Someone can use AI without leaving a trace, so `human_only` really means "no visible AI".
- A login ending with `[bot]` or `bot` is a bot. Which bots are AI agents or AI review bots was decided by hand from the distinct bot accounts printed by the code. The trace patterns in descriptions and commits were built from what we found in this data.

**Task types**
- Each PR has one main type. A PR that adds a feature and its tests counts as a feature.
- The LLM labels are correct. We did not check them by hand; this goes in the threats to validity.

## Notes for Parts 3 and 4

### New columns

No rows were removed when labelling. Each analysis picks the rows it needs.

**`pull_request_df`**

| Column | Values | Meaning |
| --- | --- | --- |
| `group` | `AI`, `Human` | Which dataset the PR comes from |
| `author_type` | `agent`, `human`, `bot`, `other_ai` | Who opened the PR |
| `ai_involvement` | see Part 2 | How much AI was involved |
| `has_human_commits` | `True`, `False`, empty | AI PR with at least one commit by a person. Empty for Codex, Claude Code and PRs with no commit data |
| `in_matched_subset` | `True`, `False` | PR from the same repositories and period as the Human PRs |
| `task_type` | `feat`, `fix`, ... | AIDev task type |
| `task_category` | `Bug fixes`, ... | Task category from the lab instructions |

**`pr_reviews_df`, `pr_comments_df`, `pr_review_comments_df`**: `actor_type` = `human`, `ai_bot` or `other_bot` (empty when the account was deleted).

### How to use them

- **Leave out** `bot` and `other_ai` PRs from the RQ1 and RQ2 comparisons.
- **Levels of AI involvement**, from none to full: `human_only` → `human_with_ai` → `agent` (split with `has_human_commits` for Copilot, Devin and Cursor). `agent_unlabeled` can be counted as `agent`.
- **Review rounds (RQ1):** count only reviews with `actor_type == "human"` that left at least one inline comment (Human-side reviews only exist in that case, so the same rule is applied to both groups). Otherwise the agent reviewing its own PR counts as review activity (the Copilot agent alone posts 6,197 reviews on its own PRs).
- **Task types:** compare merge rate and time to merge within each `task_category` too. AI PRs have more features (43% vs 33%), so a gap between AI and Human could come from the kind of work rather than from who did it.

### Why the matched subset matters

Human PRs come only from 818 repositories with more than 500 stars, created between January and June 2025. AI PRs come from 2,807 repositories with more than 100 stars, and 92% were created in May–July 2025.

Rates (merge rate, median time to merge, lines per PR) fix the fact that the AI group is bigger. They do **not** fix the fact that the groups come from different projects. The AI merge rate is 77% on all AI PRs but 60% on AI PRs from the same repositories and period as the Human PRs (Human: 85%). One user, `tamnd`, opened 8,913 Codex PRs in one personal repository, and 87% were merged. That is 27% of all AI PRs.

**Suggestion:**
- Main results on `in_matched_subset == True` (7,541 AI PRs and the Human PRs).
- Same analysis on all PRs as a check. If both agree, the result is solid. If not, say so in the discussion.
- Results per agent too (`agent` column), since the five agents behave very differently.

## Metric plan and intended table usage


| Research question | Metric or variable | Source table(s) | Join key |
| --- | --- | --- | --- |
| RQ1: development outcomes | Merge rate, time to merge pull request | `pull_request_df` (`agent` column: agent name or `Human`) | PR `id` identifies the AI/human group; `pr_id` in related tables (want to look at same repos? could do analysis of both complete datasets even if repos not 1 to 1 match in all cases) |
| RQ1: development outcomes | Issue closing rate, time to close issue | `issue_df`, `related_issue_df`, `pull_request_df` (`agent` column: agent name or `Human`) | `related_issue_df.issue_id = issue_df.id`; `related_issue_df.pr_id = pull_request.id` |
| RQ1: development outcomes | Number of review rounds, number of commits before acceptance | `pr_reviews_df`, `pr_commits_df`, `pull_request_df` | `pull_request_df.id = pr_reviews_df.pr_id = pr_commits_df.pr_id` |
| RQ1: development outcomes | Reopened issues, revised pull requests | `issue_df`, `related_issue_df`, `pr_timeline_df`, `pull_request_df` | Issues use `issue_id`; PR history uses `pull_request_df.id = pr_timeline_df.pr_id` |
| RQ2: contribution characteristics | Files changed, lines added, lines deleted, code churn, number of commits | `pr_commit_details_df`, `pr_commits_df`, `pull_request_df` | `pull_request_df.id = pr_commit_details_df.pr_id = pr_commits_df.pr_id` |
| RQ2: contribution characteristics | Types of files modified, documentation versus code, test-related changes, configuration or infrastructure changes | `pr_commit_details_df`, `pull_request_df` | `pull_request_df.id = pr_commit_details_df.pr_id` |


All tables contain both AI and Human rows. Human PRs come from repositories with more than 500 stars, while AIDev AI PRs use repositories with more than 100 stars.

## Part 3. Metrics Definition

In this section, we define and calculate the metrics required to answer our two research questions. We will create a consolidated dataframe `pr_metrics_df` that will contain one row per Pull Request with all its calculated metrics.

### RQ1: Development Outcomes
**Question:** How do AI-related contributions differ from human contributions in terms of development outcomes?

**Metrics calculated at the PR level:**
- **is_merged (Merge rate):** Boolean indicating if the PR was merged (`merged_at` is not null).
- **time_to_merge_hours:** Time elapsed between the creation of the PR and its merge (in hours).
- **issue_is_closed (Issue closing rate):** Boolean indicating if the linked issue was closed.
- **time_to_close_issue_hours:** Time elapsed between the creation and closing of the linked issue (in hours).
- **num_review_rounds:** Number of human reviews with at least one inline comment posted on the PR (from `pr_reviews_df` and `pr_review_comments_df`). Human-side reviews are only visible when they left inline comments, so the same rule is applied to both groups.
- **num_commits:** Total number of commits before acceptance (from `pr_commits_df`).
- **is_revised:** Boolean indicating if the PR was revised following comments (using `pr_timeline_df` to check for multiple review requests or reopening events).

In [ ]:
import pandas as pd
import numpy as np

# 1. Initialize the metrics dataframe with basic PR info
pr_df = dataframes['pull_request_df'].copy()
pr_df['created_at'] = pd.to_datetime(pr_df['created_at'], errors='coerce', utc=True)
pr_df['merged_at'] = pd.to_datetime(pr_df['merged_at'], errors='coerce', utc=True)

pr_metrics_df = pr_df[['id', 'group', 'agent', 'ai_involvement', 'repo_id', 'state', 'task_category', 'in_matched_subset']].copy()

# --- Merge Rate & Time to merge ---
pr_metrics_df['is_merged'] = pr_df['merged_at'].notna()
pr_metrics_df['time_to_merge_hours'] = (pr_df['merged_at'] - pr_df['created_at']).dt.total_seconds() / 3600.0

# --- Issue Closing Rate & Time to close ---
issue_df = dataframes['issue_df'].copy()
issue_df['created_at'] = pd.to_datetime(issue_df['created_at'], errors='coerce', utc=True)
issue_df['closed_at'] = pd.to_datetime(issue_df['closed_at'], errors='coerce', utc=True)
issue_df['time_to_close_issue_hours'] = (issue_df['closed_at'] - issue_df['created_at']).dt.total_seconds() / 3600.0
issue_df['issue_is_closed'] = issue_df['closed_at'].notna()

# Link issues to PRs
related_issue_df = dataframes['related_issue_df'].copy()
pr_issues = related_issue_df.merge(issue_df[['id', 'issue_is_closed', 'time_to_close_issue_hours']], 
                                   left_on='issue_id', right_on='id', how='left')
# Take the first linked issue if there are multiple
pr_issues_agg = pr_issues.groupby('pr_id').agg({
    'issue_is_closed': 'first',
    'time_to_close_issue_hours': 'first'
}).reset_index()

pr_metrics_df = pr_metrics_df.merge(pr_issues_agg, left_on='id', right_on='pr_id', how='left').drop(columns=['pr_id'])

# --- Number of review rounds ---
reviews_df = dataframes['pr_reviews_df'].copy()
# Only count human reviews as suggested in the notes (actor_type is set in Part 2).
# Human-side reviews only exist when they left inline comments, so the same rule is applied to AI PRs
review_comment_ids = dataframes['pr_review_comments_df']['pull_request_review_id']
human_reviews = reviews_df[(reviews_df['actor_type'] == 'human') & reviews_df['id'].isin(review_comment_ids)]
review_counts = human_reviews.groupby('pr_id').size().reset_index(name='num_review_rounds')
pr_metrics_df = pr_metrics_df.merge(review_counts, left_on='id', right_on='pr_id', how='left').drop(columns=['pr_id'])
pr_metrics_df['num_review_rounds'] = pr_metrics_df['num_review_rounds'].fillna(0)

# --- Number of commits ---
commits_df = dataframes['pr_commits_df'].copy()
commit_counts = commits_df.groupby('pr_id').size().reset_index(name='num_commits')
pr_metrics_df = pr_metrics_df.merge(commit_counts, left_on='id', right_on='pr_id', how='left').drop(columns=['pr_id'])
pr_metrics_df['num_commits'] = pr_metrics_df['num_commits'].fillna(0)

# --- Revised Pull Requests ---
# A PR is considered revised if it was reopened or if a review was requested multiple times
timeline_df = dataframes['pr_timeline_df'].copy()
revision_events = timeline_df[timeline_df['event'].isin(['reopened', 'review_requested'])]
revised_prs = revision_events['pr_id'].unique()
pr_metrics_df['is_revised'] = pr_metrics_df['id'].isin(revised_prs)

display(pr_metrics_df.head())

### RQ2: Contribution Characteristics
**Question:** How do AI-related and human contributions differ in terms of size, scope, and type?

**Metrics calculated at the PR level:**
- **num_files_changed:** Number of unique files modified per PR.
- **lines_added:** Total number of lines added.
- **lines_deleted:** Total number of lines deleted.
- **code_churn:** Lines added + lines deleted.
- **is_docs_change:** Boolean, true if Markdown files (`.md`, `.txt`) are modified.
- **is_test_change:** Boolean, true if test files (`test`, `.spec`) are modified.
- **is_config_change:** Boolean, true if configuration files (`.json`, `.yml`, `.yaml`, `Dockerfile`) are modified.

In [ ]:
# --- File characteristics, Lines & Churn ---
commit_details_df = dataframes['pr_commit_details_df'].copy()

# Ensure numeric values are correct
commit_details_df['additions'] = pd.to_numeric(commit_details_df['additions'], errors='coerce').fillna(0)
commit_details_df['deletions'] = pd.to_numeric(commit_details_df['deletions'], errors='coerce').fillna(0)
commit_details_df['changes'] = pd.to_numeric(commit_details_df['changes'], errors='coerce').fillna(0)

# Categorization of file types
def categorize_file(filename):
    if not isinstance(filename, str):
        return 'code'
    lower_name = filename.lower()
    if lower_name.endswith('.md') or lower_name.endswith('.txt') or 'docs/' in lower_name:
        return 'docs'
    if 'test' in lower_name or lower_name.endswith('.spec.js') or lower_name.endswith('.spec.ts'):
        return 'test'
    if lower_name.endswith(('.yml', '.yaml', '.json', '.toml', '.ini', 'dockerfile')):
        return 'config'
    return 'code'

commit_details_df['file_category'] = commit_details_df['filename'].apply(categorize_file)

# Aggregation by PR
file_metrics = commit_details_df.groupby('pr_id').agg({
    'filename': 'nunique', # Number of files changed
    'additions': 'sum',
    'deletions': 'sum',
    'changes': 'sum'       # Code churn
}).rename(columns={'filename': 'num_files_changed', 'changes': 'code_churn'})

# Check for the presence of certain types of changes
type_metrics = commit_details_df.groupby('pr_id')['file_category'].apply(lambda x: pd.Series({
    'is_docs_change': 'docs' in x.values,
    'is_test_change': 'test' in x.values,
    'is_config_change': 'config' in x.values
})).unstack().fillna(False)

# Merge with our main metrics dataframe
pr_metrics_df = pr_metrics_df.merge(file_metrics, left_on='id', right_on='pr_id', how='left')
pr_metrics_df = pr_metrics_df.merge(type_metrics, left_on='id', right_on='pr_id', how='left')

# Replace NaN with 0 for PRs without commit details
fill_zero_cols = ['num_files_changed', 'additions', 'deletions', 'code_churn']
pr_metrics_df[fill_zero_cols] = pr_metrics_df[fill_zero_cols].fillna(0)
fill_false_cols = ['is_docs_change', 'is_test_change', 'is_config_change']
pr_metrics_df[fill_false_cols] = pr_metrics_df[fill_false_cols].fillna(False)

print(f"Final size of the metrics dataframe: {pr_metrics_df.shape}")
display(pr_metrics_df.head())

# The Human group keeps only PRs opened by people: PRs opened by bots, by other AI agents,
# or by an AIDev agent that AIDev missed are left out (see ai_involvement in Part 2)
NOT_HUMAN = ['bot', 'other_ai', 'agent_unlabeled']
print("Human PRs left out:", pr_metrics_df.loc[pr_metrics_df['ai_involvement'].isin(NOT_HUMAN), 'ai_involvement'].value_counts().to_dict())
pr_metrics_df = pr_metrics_df[~pr_metrics_df['ai_involvement'].isin(NOT_HUMAN)].reset_index(drop=True)

# Save the cleaned and prepared dataset for analysis

# Save metrics for each research question separately
rq1_cols = ['id', 'group', 'agent', 'ai_involvement', 'repo_id', 'state', 'task_category', 'in_matched_subset', 'is_merged', 'time_to_merge_hours', 'issue_is_closed', 'time_to_close_issue_hours', 'num_review_rounds', 'num_commits', 'is_revised']
rq2_cols = ['id', 'group', 'agent', 'ai_involvement', 'repo_id', 'state', 'task_category', 'in_matched_subset', 'num_files_changed', 'additions', 'deletions', 'code_churn', 'is_docs_change', 'is_test_change', 'is_config_change']

rq1_df = pr_metrics_df[rq1_cols]
rq2_df = pr_metrics_df[rq2_cols]

rq1_df.to_csv('../data/rq1_metrics.csv', index=False)
rq2_df.to_csv('../data/rq2_metrics.csv', index=False)

print("Metrics have been saved to '../data/rq1_metrics.csv' and '../data/rq2_metrics.csv'")
